# Baseline Models

In [11]:
# Load modeling data and persisted project contracts

import json
import numpy as np
import pandas as pd

from pathlib import Path


DATA_PATH = Path(
    "../data/engineered/ronaldo_1000_ml_engineered.csv"
)

ARTIFACT_DIR = Path("../artifacts")


# Load engineered dataset
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

df = (
    df
    .sort_values("date")
    .reset_index(drop=True)
)


# Load saved project specifications
with open(
    ARTIFACT_DIR / "feature_spec.json",
    "r",
    encoding="utf-8"
) as f:
    feature_spec = json.load(f)

with open(
    ARTIFACT_DIR / "temporal_folds.json",
    "r",
    encoding="utf-8"
) as f:
    temporal_folds_spec = json.load(f)

with open(
    ARTIFACT_DIR / "validation_policy.json",
    "r",
    encoding="utf-8"
) as f:
    validation_policy = json.load(f)

with open(
    ARTIFACT_DIR / "preprocessing_spec.json",
    "r",
    encoding="utf-8"
) as f:
    preprocessing_spec = json.load(f)


# Restore modeling contracts
TARGET = feature_spec["primary_target"]

SECONDARY_TARGETS = (
    feature_spec["secondary_targets"]
)

FULL_FEATURES = (
    feature_spec["full_features"]
)

NUMERIC_FEATURES = (
    feature_spec["numeric_features"]
)

CATEGORICAL_FEATURES = (
    feature_spec["categorical_features"]
)

FEATURE_SETS = (
    preprocessing_spec["feature_sets"]
)


# Final untouched test boundary
FINAL_TEST_START = pd.Timestamp(
    validation_policy["final_test_start"]
)

development_df = df[
    df["date"] < FINAL_TEST_START
].copy()

final_test_df = df[
    df["date"] >= FINAL_TEST_START
].copy()


# Integrity checks
assert len(df) == 1333
assert df["match_id"].nunique() == 1333
assert df[TARGET].sum() == 978

assert len(FULL_FEATURES) == 84
assert len(FEATURE_SETS["full"]) == 84
assert len(FEATURE_SETS["context"]) == 80
assert len(FEATURE_SETS["no_season"]) == 83

assert len(temporal_folds_spec) == 4

assert len(development_df) == 1255
assert len(final_test_df) == 78

assert (
    development_df["date"].max()
    < final_test_df["date"].min()
)


print("PHASE 6 — MODELING LOAD")
print("=" * 65)

print("Full dataset:", df.shape)

print(
    "Career period:",
    df["date"].min().date(),
    "→",
    df["date"].max().date()
)

print("\nDevelopment pool:")
print("Matches:", len(development_df))
print(
    "Period:",
    development_df["date"].min().date(),
    "→",
    development_df["date"].max().date()
)
print(
    "Goals:",
    int(development_df[TARGET].sum())
)

print("\nUntouched final test:")
print("Matches:", len(final_test_df))
print(
    "Period:",
    final_test_df["date"].min().date(),
    "→",
    final_test_df["date"].max().date()
)
print(
    "Goals:",
    int(final_test_df[TARGET].sum())
)

print("\nFeature sets:")
for name, features in FEATURE_SETS.items():
    print(
        f"{name:<12}: {len(features)} features"
    )

print(
    "\nPrimary selection metric:",
    validation_policy["primary_metric"]
)

print("\nPhase 6 modeling contract: PASSED")

PHASE 6 — MODELING LOAD
Full dataset: (1333, 100)
Career period: 2002-08-14 → 2026-08-28

Development pool:
Matches: 1255
Period: 2002-08-14 → 2024-12-06
Goals: 916

Untouched final test:
Matches: 78
Period: 2025-01-09 → 2026-08-28
Goals: 62

Feature sets:
full        : 84 features
context     : 80 features
no_season   : 83 features

Primary selection metric: poisson_deviance

Phase 6 modeling contract: PASSED


## Restore the preprocessing factory

In [13]:
# Define preprocessing factory for count models

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


def build_linear_preprocessor(feature_set):
    """
    Build preprocessing for linear/count models.

    The returned transformer is unfitted.
    It must be fitted only on the training portion
    of each temporal fold.
    """

    numeric_features = [
        feature
        for feature in feature_set
        if feature in NUMERIC_FEATURES
    ]

    categorical_features = [
        feature
        for feature in feature_set
        if feature in CATEGORICAL_FEATURES
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features,
            ),
        ],
        remainder="drop",
    )

    return preprocessor


# Build but do not fit one transformer as a sanity check
preprocessor_check = build_linear_preprocessor(
    FEATURE_SETS["context"]
)


print("PREPROCESSING FACTORY CHECK")
print("=" * 60)

print(
    "Feature set:",
    "context"
)

print(
    "Raw features:",
    len(FEATURE_SETS["context"])
)

print(
    "Transformer fitted:",
    hasattr(
        preprocessor_check,
        "transformers_"
    )
)


# Confirm saved preprocessing contract agrees
# with the executable implementation.
assert (
    preprocessing_spec["preprocessing_strategy"]
    ["linear_count_models"]["numeric"]
    == "StandardScaler"
)

assert "OneHotEncoder" in (
    preprocessing_spec["preprocessing_strategy"]
    ["linear_count_models"]["categorical"]
)

assert not hasattr(
    preprocessor_check,
    "transformers_"
)


print("\nPreprocessing factory integrity: PASSED")
print("No data has been fitted.")

PREPROCESSING FACTORY CHECK
Feature set: context
Raw features: 80
Transformer fitted: False

Preprocessing factory integrity: PASSED
No data has been fitted.


## Build temporal model evaluation helper

In [15]:
# Build leakage-safe temporal evaluation helper

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
)


def evaluate_goal_predictions(y_true, y_pred):
    """
    Evaluate expected-goals predictions.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_pred_safe = np.clip(
        y_pred,
        1e-9,
        None
    )

    return {
        "poisson_deviance": mean_poisson_deviance(
            y_true,
            y_pred_safe
        ),
        "mae": mean_absolute_error(
            y_true,
            y_pred_safe
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred_safe
            )
        ),
        "actual_mean_goals": y_true.mean(),
        "predicted_mean_goals": y_pred_safe.mean(),
        "mean_bias": (
            y_pred_safe.mean()
            - y_true.mean()
        ),
    }


def evaluate_temporal_model(
    model_factory,
    feature_set_name,
):
    """
    Evaluate a model across the four saved expanding-window folds.

    model_factory must return a fresh unfitted sklearn-compatible
    estimator or pipeline each time it is called.
    """

    features = FEATURE_SETS[
        feature_set_name
    ]

    fold_results = []
    oof_parts = []

    for fold in temporal_folds_spec:

        train_start = pd.Timestamp(
            fold["train_start"]
        )
        train_end = pd.Timestamp(
            fold["train_end"]
        )

        val_start = pd.Timestamp(
            fold["validation_start"]
        )
        val_end = pd.Timestamp(
            fold["validation_end"]
        )


        train = df[
            (df["date"] >= train_start)
            & (df["date"] <= train_end)
        ].copy()

        val = df[
            (df["date"] >= val_start)
            & (df["date"] <= val_end)
        ].copy()


        X_train = train[features]
        y_train = train[TARGET]

        X_val = val[features]
        y_val = val[TARGET]


        # Fresh model for every fold
        model = model_factory()

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_val
        )

        metrics = evaluate_goal_predictions(
            y_val,
            y_pred
        )


        fold_results.append({
            "fold": fold["name"],
            "train_matches": len(train),
            "val_matches": len(val),
            **metrics,
        })


        oof_parts.append(
            pd.DataFrame({
                "match_id": val["match_id"].values,
                "date": val["date"].values,
                "fold": fold["name"],
                "y_true": y_val.values,
                "y_pred": y_pred,
            })
        )


    fold_results = pd.DataFrame(
        fold_results
    )

    oof = (
        pd.concat(
            oof_parts,
            ignore_index=True
        )
        .sort_values("date")
        .reset_index(drop=True)
    )


    macro_metrics = (
        fold_results[
            [
                "poisson_deviance",
                "mae",
                "rmse",
                "mean_bias",
            ]
        ]
        .mean()
        .to_dict()
    )


    pooled_metrics = evaluate_goal_predictions(
        oof["y_true"],
        oof["y_pred"]
    )


    return {
        "feature_set": feature_set_name,
        "fold_results": fold_results,
        "oof": oof,
        "macro_metrics": macro_metrics,
        "pooled_metrics": pooled_metrics,
    }


print("TEMPORAL EVALUATION HELPER")
print("=" * 60)

print(
    "Saved folds:",
    len(temporal_folds_spec)
)

print(
    "Expected OOF matches:",
    sum(
        fold["validation_matches"]
        for fold in temporal_folds_spec
    )
)

assert len(temporal_folds_spec) == 4

assert sum(
    fold["validation_matches"]
    for fold in temporal_folds_spec
) == 340

print("\nTemporal evaluation helper integrity: PASSED")

TEMPORAL EVALUATION HELPER
Saved folds: 4
Expected OOF matches: 340

Temporal evaluation helper integrity: PASSED


## Poisson regression on the context feature set

In [19]:
# Poisson regression baseline on context features

from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline


def make_poisson_context_model():
    """
    Fresh Poisson regression pipeline for each temporal fold.
    """

    preprocessor = build_linear_preprocessor(
        FEATURE_SETS["context"]
    )

    model = PoissonRegressor(
        alpha=1.0,
        max_iter=2000
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


poisson_context_results = evaluate_temporal_model(
    model_factory=make_poisson_context_model,
    feature_set_name="context",
)


print("POISSON REGRESSION — CONTEXT FEATURES")
print("=" * 100)

print(
    poisson_context_results["fold_results"][
        [
            "fold",
            "train_matches",
            "val_matches",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


print("\nMACRO VALIDATION")
print("=" * 50)

for metric, value in (
    poisson_context_results["macro_metrics"].items()
):
    print(f"{metric:<25} {value:.6f}")


print("\nPOOLED OOF VALIDATION")
print("=" * 50)

for metric, value in (
    poisson_context_results["pooled_metrics"].items()
):
    print(f"{metric:<25} {value:.6f}")


print("\nBASELINE COMPARISON")
print("=" * 70)

print(
    "Historical mean pooled deviance : 1.106694"
)

print(
    "Recent-20 pooled deviance       : 1.141576"
)

print(
    "Poisson context pooled deviance :",
    round(
        poisson_context_results[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

POISSON REGRESSION — CONTEXT FEATURES
  fold  train_matches  val_matches  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance    mae   rmse
Fold 1            915          158             0.7911                1.0469     0.2557            1.1452 0.7369 0.9063
Fold 2           1073           72             0.5000                0.8237     0.3237            1.2807 0.7369 0.8514
Fold 3           1145           59             0.9153                0.7993    -0.1160            1.1717 0.7319 0.9549
Fold 4           1204           51             0.8431                0.9258     0.0827            0.9775 0.6550 0.8353

MACRO VALIDATION
poisson_deviance          1.143774
mae                       0.715180
rmse                      0.886990
mean_bias                 0.136537

POOLED OOF VALIDATION
poisson_deviance          1.153344
mae                       0.723756
rmse                      0.893384
actual_mean_goals         0.758824
predicted_mean_goals      0.938492
mean_bias  

In [21]:
# Compare Poisson across feature-set variants

def make_poisson_model(feature_set_name):
    """
    Create a fresh Poisson pipeline for a specified feature set.
    """

    features = FEATURE_SETS[
        feature_set_name
    ]

    return Pipeline(
        steps=[
            (
                "preprocessor",
                build_linear_preprocessor(features)
            ),
            (
                "model",
                PoissonRegressor(
                    alpha=1.0,
                    max_iter=2000
                )
            ),
        ]
    )


poisson_feature_results = {}
comparison_rows = []


for feature_set_name in [
    "context",
    "no_season",
    "full",
]:

    result = evaluate_temporal_model(
        model_factory=lambda name=feature_set_name: (
            make_poisson_model(name)
        ),
        feature_set_name=feature_set_name,
    )

    poisson_feature_results[
        feature_set_name
    ] = result


    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    comparison_rows.append({
        "feature_set": feature_set_name,

        "macro_poisson_deviance":
            macro["poisson_deviance"],

        "pooled_poisson_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],
    })


poisson_feature_comparison = pd.DataFrame(
    comparison_rows
).sort_values(
    "macro_poisson_deviance"
)


print("POISSON FEATURE-SET COMPARISON — ALPHA = 1.0")
print("=" * 95)

print(
    poisson_feature_comparison
    .round(6)
    .to_string(index=False)
)


print("\nFOLD POISSON DEVIANCE")
print("=" * 75)

fold_comparison = pd.DataFrame({
    name: result["fold_results"]
        .set_index("fold")["poisson_deviance"]

    for name, result
    in poisson_feature_results.items()
})

print(
    fold_comparison
    .round(6)
    .to_string()
)


print("\nREFERENCE BASELINES")
print("=" * 75)
print(
    "Historical mean — macro deviance : 1.115870"
)
print(
    "Historical mean — pooled deviance: 1.106694"
)
print(
    "Recent-20      — macro deviance : 1.154657"
)
print(
    "Recent-20      — pooled deviance: 1.141576"
)

POISSON FEATURE-SET COMPARISON — ALPHA = 1.0
feature_set  macro_poisson_deviance  pooled_poisson_deviance  pooled_mae  pooled_rmse  pooled_bias
  no_season                1.137122                 1.144218    0.716981     0.887783     0.163090
       full                1.137849                 1.144813    0.717227     0.888034     0.163893
    context                1.143774                 1.153344    0.723756     0.893384     0.179668

FOLD POISSON DEVIANCE
         context  no_season      full
fold                                 
Fold 1  1.145221   1.130212  1.130065
Fold 2  1.280709   1.275401  1.278690
Fold 3  1.171708   1.168153  1.168225
Fold 4  0.977457   0.974721  0.974415

REFERENCE BASELINES
Historical mean — macro deviance : 1.115870
Historical mean — pooled deviance: 1.106694
Recent-20      — macro deviance : 1.154657
Recent-20      — pooled deviance: 1.141576


In [23]:
# Tune Poisson regularization using temporal validation

POISSON_ALPHAS = [
    0.001,
    0.01,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0,
]

poisson_alpha_results = []
poisson_alpha_details = {}


for alpha in POISSON_ALPHAS:

    def make_model(alpha_value=alpha):

        return Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_linear_preprocessor(
                        FEATURE_SETS["no_season"]
                    )
                ),
                (
                    "model",
                    PoissonRegressor(
                        alpha=alpha_value,
                        max_iter=3000
                    )
                ),
            ]
        )


    result = evaluate_temporal_model(
        model_factory=make_model,
        feature_set_name="no_season",
    )

    poisson_alpha_details[alpha] = result

    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_deviances = (
        result["fold_results"]
        .set_index("fold")["poisson_deviance"]
    )


    poisson_alpha_results.append({
        "alpha": alpha,

        "macro_poisson_deviance":
            macro["poisson_deviance"],

        "pooled_poisson_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1_deviance":
            fold_deviances["Fold 1"],

        "fold2_deviance":
            fold_deviances["Fold 2"],

        "fold3_deviance":
            fold_deviances["Fold 3"],

        "fold4_deviance":
            fold_deviances["Fold 4"],
    })


poisson_alpha_results = (
    pd.DataFrame(poisson_alpha_results)
    .sort_values("macro_poisson_deviance")
    .reset_index(drop=True)
)


print("POISSON REGULARIZATION SEARCH — NO_SEASON")
print("=" * 120)

print(
    poisson_alpha_results
    .round(6)
    .to_string(index=False)
)


best_alpha = poisson_alpha_results.loc[
    0,
    "alpha"
]

best_poisson_result = (
    poisson_alpha_details[best_alpha]
)


print("\nBEST ALPHA")
print("=" * 60)

print("Alpha:", best_alpha)

print(
    "Macro Poisson deviance:",
    round(
        best_poisson_result[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled Poisson deviance:",
    round(
        best_poisson_result[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        best_poisson_result[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        best_poisson_result[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nREFERENCE")
print("=" * 60)
print("Historical mean macro : 1.115870")
print("Historical mean pooled: 1.106694")
print("Recent-20 macro       : 1.154657")
print("Recent-20 pooled      : 1.141576")

POISSON REGULARIZATION SEARCH — NO_SEASON
  alpha  macro_poisson_deviance  pooled_poisson_deviance  pooled_mae  pooled_rmse  pooled_bias  fold1_deviance  fold2_deviance  fold3_deviance  fold4_deviance
 30.000                1.114214                 1.105473    0.707677     0.865734     0.011153        1.057363        1.252396        1.140365        1.006732
100.000                1.114363                 1.105096    0.711042     0.865555    -0.018190        1.057136        1.246089        1.140163        1.014062
 10.000                1.117580                 1.110723    0.701311     0.868278     0.053539        1.065791        1.259822        1.145479        0.999226
  3.000                1.125330                 1.123388    0.699313     0.875112     0.108987        1.089105        1.266935        1.155618        0.989662
  1.000                1.137122                 1.144218    0.716981     0.887783     0.163090        1.130212        1.275401        1.168153        0.974721
  0.

In [25]:
# Refine Poisson regularization around the best region

REFINED_ALPHAS = [
    15.0,
    20.0,
    25.0,
    30.0,
    40.0,
    50.0,
    60.0,
    75.0,
    100.0,
    150.0,
    200.0,
]

refined_results = []
refined_details = {}


for alpha in REFINED_ALPHAS:

    def make_model(alpha_value=alpha):

        return Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_linear_preprocessor(
                        FEATURE_SETS["no_season"]
                    )
                ),
                (
                    "model",
                    PoissonRegressor(
                        alpha=alpha_value,
                        max_iter=3000
                    )
                ),
            ]
        )


    result = evaluate_temporal_model(
        model_factory=make_model,
        feature_set_name="no_season",
    )

    refined_details[alpha] = result

    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")["poisson_deviance"]
    )


    refined_results.append({
        "alpha": alpha,
        "macro_poisson_deviance":
            macro["poisson_deviance"],
        "pooled_poisson_deviance":
            pooled["poisson_deviance"],
        "pooled_mae":
            pooled["mae"],
        "pooled_rmse":
            pooled["rmse"],
        "pooled_bias":
            pooled["mean_bias"],
        "fold1":
            fold_dev["Fold 1"],
        "fold2":
            fold_dev["Fold 2"],
        "fold3":
            fold_dev["Fold 3"],
        "fold4":
            fold_dev["Fold 4"],
    })


refined_results = (
    pd.DataFrame(refined_results)
    .sort_values("macro_poisson_deviance")
    .reset_index(drop=True)
)


print("REFINED POISSON ALPHA SEARCH")
print("=" * 120)

print(
    refined_results
    .round(6)
    .to_string(index=False)
)


BEST_POISSON_ALPHA = float(
    refined_results.loc[0, "alpha"]
)

BEST_POISSON_RESULT = (
    refined_details[BEST_POISSON_ALPHA]
)


print("\nSELECTED POISSON CONFIGURATION")
print("=" * 65)

print(
    "Feature set : no_season"
)

print(
    "Alpha       :",
    BEST_POISSON_ALPHA
)

print(
    "Macro dev.  :",
    round(
        BEST_POISSON_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled dev. :",
    round(
        BEST_POISSON_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE  :",
    round(
        BEST_POISSON_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias :",
    round(
        BEST_POISSON_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)

REFINED POISSON ALPHA SEARCH
 alpha  macro_poisson_deviance  pooled_poisson_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
  50.0                1.113978                 1.104881    0.709464     0.865464    -0.003735 1.056583 1.249267 1.139866 1.010196
  40.0                1.114001                 1.105028    0.708760     0.865531     0.002312 1.056745 1.250562 1.139974 1.008724
  60.0                1.114033                 1.104864    0.709964     0.865455    -0.008162 1.056609 1.248322 1.139874 1.011328
  75.0                1.114148                 1.104922    0.710486     0.865480    -0.012955 1.056777 1.247257 1.139956 1.012602
  30.0                1.114214                 1.105473    0.707677     0.865734     0.011153 1.057363 1.252396 1.140365 1.006732
 100.0                1.114363                 1.105096    0.711042     0.865555    -0.018190 1.057136 1.246089 1.140163 1.014062
  25.0                1.114477                 1.105931    0.

## Negative Binomial baseline

In [27]:
# Prepare Negative Binomial modeling path

import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


def build_statsmodels_preprocessor(feature_set):
    """
    Dense preprocessing for statsmodels count models.

    This remains unfitted until called inside
    an individual temporal training fold.
    """

    numeric_features = [
        feature
        for feature in feature_set
        if feature in NUMERIC_FEATURES
    ]

    categorical_features = [
        feature
        for feature in feature_set
        if feature in CATEGORICAL_FEATURES
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_features,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
    )

    return preprocessor


nb_preprocessor_check = (
    build_statsmodels_preprocessor(
        FEATURE_SETS["no_season"]
    )
)


print("NEGATIVE BINOMIAL SETUP")
print("=" * 60)

print(
    "Statsmodels version:",
    sm.__version__
    if hasattr(sm, "__version__")
    else "available"
)

print(
    "Feature set:",
    "no_season"
)

print(
    "Raw features:",
    len(FEATURE_SETS["no_season"])
)

print(
    "Preprocessor fitted:",
    hasattr(
        nb_preprocessor_check,
        "transformers_"
    )
)

assert not hasattr(
    nb_preprocessor_check,
    "transformers_"
)

print("\nNegative Binomial setup: PASSED")
print("No model or preprocessor has been fitted.")

NEGATIVE BINOMIAL SETUP
Statsmodels version: 0.14.2
Feature set: no_season
Raw features: 83
Preprocessor fitted: False

Negative Binomial setup: PASSED
No model or preprocessor has been fitted.


In [ ]:
# Temporal Negative Binomial baseline

def evaluate_negative_binomial(
    dispersion_alpha=0.1,
    feature_set_name="no_season",
):
    """
    Evaluate statsmodels Negative Binomial GLM
    across the saved temporal folds.

    dispersion_alpha controls the NB2 variance:

        Var(Y) = mu + alpha * mu^2
    """

    features = FEATURE_SETS[
        feature_set_name
    ]

    fold_results = []
    oof_parts = []


    for fold in temporal_folds_spec:

        train_start = pd.Timestamp(
            fold["train_start"]
        )
        train_end = pd.Timestamp(
            fold["train_end"]
        )

        val_start = pd.Timestamp(
            fold["validation_start"]
        )
        val_end = pd.Timestamp(
            fold["validation_end"]
        )


        train = df[
            (df["date"] >= train_start)
            & (df["date"] <= train_end)
        ].copy()

        val = df[
            (df["date"] >= val_start)
            & (df["date"] <= val_end)
        ].copy()


        # Fresh preprocessing for every fold
        preprocessor = (
            build_statsmodels_preprocessor(
                features
            )
        )


        X_train = preprocessor.fit_transform(
            train[features]
        )

        X_val = preprocessor.transform(
            val[features]
        )


        # Add intercept after preprocessing
        X_train = sm.add_constant(
            X_train,
            has_constant="add"
        )

        X_val = sm.add_constant(
            X_val,
            has_constant="add"
        )


        y_train = train[TARGET].to_numpy(
            dtype=float
        )

        y_val = val[TARGET].to_numpy(
            dtype=float
        )


        nb_model = sm.GLM(
            y_train,
            X_train,
            family=sm.families.NegativeBinomial(
                alpha=dispersion_alpha
            ),
        )


        nb_result = nb_model.fit(
            maxiter=200,
            disp=0
        )


        y_pred = nb_result.predict(
            X_val
        )


        metrics = evaluate_goal_predictions(
            y_val,
            y_pred
        )


        fold_results.append({
            "fold": fold["name"],
            "train_matches": len(train),
            "val_matches": len(val),
            "converged": bool(
                nb_result.converged
            ),
            **metrics,
        })


        oof_parts.append(
            pd.DataFrame({
                "match_id":
                    val["match_id"].values,

                "date":
                    val["date"].values,

                "fold":
                    fold["name"],

                "y_true":
                    y_val,

                "y_pred":
                    y_pred,
            })
        )


    fold_results = pd.DataFrame(
        fold_results
    )

    oof = (
        pd.concat(
            oof_parts,
            ignore_index=True
        )
        .sort_values("date")
        .reset_index(drop=True)
    )


    macro_metrics = (
        fold_results[
            [
                "poisson_deviance",
                "mae",
                "rmse",
                "mean_bias",
            ]
        ]
        .mean()
        .to_dict()
    )


    pooled_metrics = (
        evaluate_goal_predictions(
            oof["y_true"],
            oof["y_pred"]
        )
    )


    return {
        "dispersion_alpha":
            dispersion_alpha,

        "feature_set":
            feature_set_name,

        "fold_results":
            fold_results,

        "oof":
            oof,

        "macro_metrics":
            macro_metrics,

        "pooled_metrics":
            pooled_metrics,
    }


nb_initial_result = evaluate_negative_binomial(
    dispersion_alpha=0.1,
    feature_set_name="no_season",
)


print("NEGATIVE BINOMIAL — INITIAL TEMPORAL TEST")
print("=" * 100)

print(
    nb_initial_result["fold_results"][
        [
            "fold",
            "converged",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\nMACRO VALIDATION")
print("=" * 50)

for metric, value in (
    nb_initial_result[
        "macro_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nPOOLED OOF VALIDATION")
print("=" * 50)

for metric, value in (
    nb_initial_result[
        "pooled_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


assert len(
    nb_initial_result["oof"]
) == 340

assert nb_initial_result[
    "fold_results"
]["converged"].all()


print("\nAll four NB fits converged: PASSED")

In [31]:
# Build compact feature set for statistical count models

COMPACT_NUMERIC_FEATURES = [
    # Career / ageing state
    "age_years",
    "career_goals_per_game_pre",

    # Recent scoring form
    "goals_per_game_last_20",
    "goals_ewma_pre",
    "goalless_streak_games_pre",

    # Schedule / recovery
    "days_rest",

    # Ronaldo-team scoring environment
    "team_ronaldo_goals_per_game_pre",
    "team_goals_for_last_5_scored_matches_pre",

    # Opponent-specific history
    "ronaldo_goals_per_game_vs_opponent_pre",

    # Matchup context
    "matchup_elo_diff_pre",
    "opponent_h2h_points_per_game_pre",
    "opponent_goals_against_per_h2h_scored_match_pre",

    # Match importance
    "competition_importance",

    # Missing / novelty state
    "first_time_opponent_pre",
    "no_h2h_score_history_pre",
]


COMPACT_CATEGORICAL_FEATURES = [
    "competition_type",
    "venue",
]


COMPACT_FEATURES = (
    COMPACT_NUMERIC_FEATURES
    + COMPACT_CATEGORICAL_FEATURES
)


# Check that every requested feature really exists
missing_compact_features = [
    feature
    for feature in COMPACT_FEATURES
    if feature not in df.columns
]


print("COMPACT STATISTICAL FEATURE SET")
print("=" * 70)

print("Numeric features     :", len(COMPACT_NUMERIC_FEATURES))
print("Categorical features :", len(COMPACT_CATEGORICAL_FEATURES))
print("Total raw features   :", len(COMPACT_FEATURES))
print("Missing features     :", missing_compact_features)


assert len(missing_compact_features) == 0


# Use Fold 1 purely to inspect the transformed design matrix
fold = temporal_folds_spec[0]

train_start = pd.Timestamp(fold["train_start"])
train_end = pd.Timestamp(fold["train_end"])

val_start = pd.Timestamp(fold["validation_start"])
val_end = pd.Timestamp(fold["validation_end"])


train = df[
    (df["date"] >= train_start)
    & (df["date"] <= train_end)
].copy()

val = df[
    (df["date"] >= val_start)
    & (df["date"] <= val_end)
].copy()


compact_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            COMPACT_NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                drop="first",
            ),
            COMPACT_CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    sparse_threshold=0,
)


X_train_compact = compact_preprocessor.fit_transform(
    train[COMPACT_FEATURES]
)

X_val_compact = compact_preprocessor.transform(
    val[COMPACT_FEATURES]
)


X_train_compact = sm.add_constant(
    X_train_compact,
    has_constant="add",
)

X_val_compact = sm.add_constant(
    X_val_compact,
    has_constant="add",
)


matrix_rank = np.linalg.matrix_rank(
    X_train_compact
)

condition_number = np.linalg.cond(
    X_train_compact
)


print("\nFOLD 1 DESIGN MATRIX")
print("=" * 70)

print("Train shape      :", X_train_compact.shape)
print("Validation shape :", X_val_compact.shape)
print("Matrix rank      :", matrix_rank)
print(
    "Full column rank :",
    matrix_rank == X_train_compact.shape[1]
)
print(
    "Condition number :",
    round(condition_number, 2)
)

print(
    "Finite train     :",
    np.isfinite(X_train_compact).all()
)
print(
    "Finite validation:",
    np.isfinite(X_val_compact).all()
)


assert np.isfinite(
    X_train_compact
).all()

assert np.isfinite(
    X_val_compact
).all()


print(
    "\nCompact statistical preprocessing: PASSED"
)

COMPACT STATISTICAL FEATURE SET
Numeric features     : 15
Categorical features : 2
Total raw features   : 17
Missing features     : []

FOLD 1 DESIGN MATRIX
Train shape      : (915, 21)
Validation shape : (158, 21)
Matrix rank      : 21
Full column rank : True
Condition number : 18.25
Finite train     : True
Finite validation: True

Compact statistical preprocessing: PASSED


In [33]:
# Compact Negative Binomial temporal baseline

def build_compact_statsmodels_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                COMPACT_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    drop="first",
                ),
                COMPACT_CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
    )


def evaluate_compact_negative_binomial(
    dispersion_alpha=0.1,
):
    fold_results = []
    oof_parts = []

    for fold in temporal_folds_spec:

        train_start = pd.Timestamp(
            fold["train_start"]
        )
        train_end = pd.Timestamp(
            fold["train_end"]
        )

        val_start = pd.Timestamp(
            fold["validation_start"]
        )
        val_end = pd.Timestamp(
            fold["validation_end"]
        )


        train = df[
            (df["date"] >= train_start)
            & (df["date"] <= train_end)
        ].copy()

        val = df[
            (df["date"] >= val_start)
            & (df["date"] <= val_end)
        ].copy()


        preprocessor = (
            build_compact_statsmodels_preprocessor()
        )


        X_train = preprocessor.fit_transform(
            train[COMPACT_FEATURES]
        )

        X_val = preprocessor.transform(
            val[COMPACT_FEATURES]
        )


        X_train = sm.add_constant(
            X_train,
            has_constant="add",
        )

        X_val = sm.add_constant(
            X_val,
            has_constant="add",
        )


        y_train = train[TARGET].to_numpy(
            dtype=float
        )

        y_val = val[TARGET].to_numpy(
            dtype=float
        )


        model = sm.GLM(
            y_train,
            X_train,
            family=sm.families.NegativeBinomial(
                alpha=dispersion_alpha
            ),
        )


        result = model.fit(
            maxiter=300,
            disp=0,
        )


        y_pred = result.predict(
            X_val
        )


        metrics = evaluate_goal_predictions(
            y_val,
            y_pred
        )


        fold_results.append({
            "fold": fold["name"],
            "train_matches": len(train),
            "val_matches": len(val),
            "converged": bool(
                result.converged
            ),
            **metrics,
        })


        oof_parts.append(
            pd.DataFrame({
                "match_id":
                    val["match_id"].values,

                "date":
                    val["date"].values,

                "fold":
                    fold["name"],

                "y_true":
                    y_val,

                "y_pred":
                    y_pred,
            })
        )


    fold_results = pd.DataFrame(
        fold_results
    )


    oof = (
        pd.concat(
            oof_parts,
            ignore_index=True
        )
        .sort_values("date")
        .reset_index(drop=True)
    )


    macro_metrics = (
        fold_results[
            [
                "poisson_deviance",
                "mae",
                "rmse",
                "mean_bias",
            ]
        ]
        .mean()
        .to_dict()
    )


    pooled_metrics = (
        evaluate_goal_predictions(
            oof["y_true"],
            oof["y_pred"]
        )
    )


    return {
        "dispersion_alpha":
            dispersion_alpha,
        "fold_results":
            fold_results,
        "oof":
            oof,
        "macro_metrics":
            macro_metrics,
        "pooled_metrics":
            pooled_metrics,
    }


nb_compact_result = (
    evaluate_compact_negative_binomial(
        dispersion_alpha=0.1
    )
)


print("COMPACT NEGATIVE BINOMIAL — ALPHA = 0.1")
print("=" * 100)

print(
    nb_compact_result["fold_results"][
        [
            "fold",
            "converged",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\nMACRO VALIDATION")
print("=" * 50)

for metric, value in (
    nb_compact_result[
        "macro_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nPOOLED OOF VALIDATION")
print("=" * 50)

for metric, value in (
    nb_compact_result[
        "pooled_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nREFERENCE")
print("=" * 50)
print("Poisson best macro  : 1.113978")
print("Poisson best pooled : 1.104881")
print("Historical macro    : 1.115870")
print("Historical pooled   : 1.106694")


assert len(
    nb_compact_result["oof"]
) == 340

assert nb_compact_result[
    "fold_results"
]["converged"].all()


print(
    "\nAll compact NB folds converged: PASSED"
)

COMPACT NEGATIVE BINOMIAL — ALPHA = 0.1
  fold  converged  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance      mae     rmse
Fold 1       True           0.791139              0.955822   0.164682          1.107051 0.707867 0.890411
Fold 2       True           0.500000              0.781594   0.281594          1.293916 0.771356 0.871055
Fold 3       True           0.915254              0.703917  -0.211337          1.302757 0.767049 0.989610
Fold 4       True           0.843137              0.984106   0.140969          0.986675 0.657891 0.840545

MACRO VALIDATION
poisson_deviance          1.172600
mae                       0.726041
rmse                      0.897905
mean_bias                 0.093977

POOLED OOF VALIDATION
poisson_deviance          1.162527
mae                       0.724085
rmse                      0.897233
actual_mean_goals         0.758824
predicted_mean_goals      0.879456
mean_bias                 0.120633

REFERENCE
Poisson best macro  : 1.1139

In [35]:
# Negative Binomial dispersion sensitivity

NB_DISPERSIONS = [
    0.01,
    0.03,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
    0.75,
    1.00,
]

nb_dispersion_rows = []
nb_dispersion_details = {}


for dispersion in NB_DISPERSIONS:

    result = evaluate_compact_negative_binomial(
        dispersion_alpha=dispersion
    )

    nb_dispersion_details[dispersion] = result

    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")["poisson_deviance"]
    )


    nb_dispersion_rows.append({
        "dispersion": dispersion,

        "macro_poisson_deviance":
            macro["poisson_deviance"],

        "pooled_poisson_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


nb_dispersion_results = (
    pd.DataFrame(nb_dispersion_rows)
    .sort_values("macro_poisson_deviance")
    .reset_index(drop=True)
)


print("NEGATIVE BINOMIAL DISPERSION SEARCH")
print("=" * 120)

print(
    nb_dispersion_results
    .round(6)
    .to_string(index=False)
)


BEST_NB_DISPERSION = float(
    nb_dispersion_results.loc[
        0,
        "dispersion"
    ]
)

BEST_NB_RESULT = (
    nb_dispersion_details[
        BEST_NB_DISPERSION
    ]
)


print("\nBEST NEGATIVE BINOMIAL CONFIGURATION")
print("=" * 65)

print(
    "Dispersion  :",
    BEST_NB_DISPERSION
)

print(
    "Macro dev.  :",
    round(
        BEST_NB_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled dev. :",
    round(
        BEST_NB_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE  :",
    round(
        BEST_NB_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias :",
    round(
        BEST_NB_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nCURRENT BENCHMARKS")
print("=" * 65)

print(
    "Regularized Poisson macro  : 1.113978"
)

print(
    "Regularized Poisson pooled : 1.104881"
)

print(
    "Historical mean macro      : 1.115870"
)

print(
    "Historical mean pooled     : 1.106694"
)

print(
    "Recent-20 macro            : 1.154657"
)

print(
    "Recent-20 pooled           : 1.141576"
)

NEGATIVE BINOMIAL DISPERSION SEARCH
 dispersion  macro_poisson_deviance  pooled_poisson_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
       0.01                1.170607                 1.159367    0.721652     0.895045     0.111462 1.100879 1.293202 1.303695 0.984652
       0.03                1.171046                 1.160068    0.722175     0.895531     0.113576 1.102259 1.293351 1.303462 0.985111
       0.05                1.171487                 1.160770    0.722702     0.896017     0.115645 1.103635 1.293505 1.303244 0.985564
       0.10                1.172600                 1.162527    0.724085     0.897233     0.120633 1.107051 1.293916 1.302757 0.986675
       0.20                1.174840                 1.166014    0.726974     0.899648     0.129891 1.113741 1.294822 1.301989 0.988809
       0.30                1.177068                 1.169430    0.729883     0.902019     0.138318 1.120195 1.295809 1.301429 0.990841
       0.50        

## Inspect Poisson fold stability and calibration

In [38]:
# Diagnose selected Poisson baseline across temporal regimes

poisson_oof = (
    BEST_POISSON_RESULT["oof"]
    .copy()
)

poisson_folds = (
    BEST_POISSON_RESULT["fold_results"]
    .copy()
)


# Absolute prediction error
poisson_oof["absolute_error"] = (
    poisson_oof["y_true"]
    - poisson_oof["y_pred"]
).abs()


# Prediction bins based only on predicted expected goals
poisson_oof["prediction_band"] = pd.cut(
    poisson_oof["y_pred"],
    bins=[
        -np.inf,
        0.40,
        0.60,
        0.80,
        1.00,
        np.inf,
    ],
    labels=[
        "<0.40",
        "0.40–0.60",
        "0.60–0.80",
        "0.80–1.00",
        "1.00+",
    ],
)


calibration_table = (
    poisson_oof
    .groupby(
        "prediction_band",
        observed=True
    )
    .agg(
        matches=("y_true", "size"),
        predicted_mean=("y_pred", "mean"),
        actual_mean=("y_true", "mean"),
        actual_scoring_rate=(
            "y_true",
            lambda x: (x >= 1).mean()
        ),
        actual_multi_goal_rate=(
            "y_true",
            lambda x: (x >= 2).mean()
        ),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)


calibration_table["mean_gap"] = (
    calibration_table["predicted_mean"]
    - calibration_table["actual_mean"]
)


print("SELECTED POISSON — FOLD STABILITY")
print("=" * 100)

print(
    poisson_folds[
        [
            "fold",
            "train_matches",
            "val_matches",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\nPREDICTED-GOAL CALIBRATION BANDS")
print("=" * 100)

print(
    calibration_table
    .round(6)
    .to_string(index=False)
)


print("\nOOF PREDICTION RANGE")
print("=" * 60)

print(
    "Minimum prediction:",
    round(
        poisson_oof["y_pred"].min(),
        6
    )
)

print(
    "Median prediction :",
    round(
        poisson_oof["y_pred"].median(),
        6
    )
)

print(
    "Maximum prediction:",
    round(
        poisson_oof["y_pred"].max(),
        6
    )
)


print("\nEXTREME PREDICTION COUNTS")
print("=" * 60)

print(
    "Predictions < 0.25:",
    int(
        (poisson_oof["y_pred"] < 0.25).sum()
    )
)

print(
    "Predictions > 1.25:",
    int(
        (poisson_oof["y_pred"] > 1.25).sum()
    )
)

print(
    "Predictions > 1.50:",
    int(
        (poisson_oof["y_pred"] > 1.50).sum()
    )
)

SELECTED POISSON — FOLD STABILITY
  fold  train_matches  val_matches  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance      mae     rmse
Fold 1            915          158           0.791139              0.765788  -0.025351          1.056583 0.694056 0.851410
Fold 2           1073           72           0.500000              0.749663   0.249663          1.249267 0.727276 0.835044
Fold 3           1145           59           0.915254              0.729552  -0.185703          1.139866 0.753995 0.946392
Fold 4           1204           51           0.843137              0.759145  -0.083992          1.010196 0.680538 0.852747

PREDICTED-GOAL CALIBRATION BANDS
prediction_band  matches  predicted_mean  actual_mean  actual_scoring_rate  actual_multi_goal_rate      mae  mean_gap
      0.60–0.80      330        0.753633     0.766667             0.536364                0.178788 0.710921 -0.013033
      0.80–1.00       10        0.803125     0.500000             0.400000       

In [40]:
# Persist Phase 6 baseline results and selected configuration

import json
from pathlib import Path

ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


baseline_model_spec = {
    "phase": "Phase 6 — Baseline Models",

    "selection_metric": "macro_poisson_deviance",

    "selected_statistical_baseline": {
        "model": "PoissonRegressor",
        "feature_set": "no_season",
        "alpha": float(BEST_POISSON_ALPHA),
        "max_iter": 3000,

        "macro_metrics": {
            key: float(value)
            for key, value
            in BEST_POISSON_RESULT[
                "macro_metrics"
            ].items()
        },

        "pooled_metrics": {
            key: float(value)
            for key, value
            in BEST_POISSON_RESULT[
                "pooled_metrics"
            ].items()
        },

        "oof_prediction_min": float(
            BEST_POISSON_RESULT[
                "oof"
            ]["y_pred"].min()
        ),

        "oof_prediction_max": float(
            BEST_POISSON_RESULT[
                "oof"
            ]["y_pred"].max()
        ),
    },

    "reference_baselines": {
        "historical_mean": {
            "macro_poisson_deviance": 1.115870,
            "pooled_poisson_deviance": 1.106694,
        },

        "recent_20": {
            "macro_poisson_deviance": 1.154657,
            "pooled_poisson_deviance": 1.141576,
        },
    },

    "negative_binomial": {
        "status": "rejected_as_primary_baseline",
        "feature_set": "compact_statistical",
        "best_dispersion_alpha": float(
            BEST_NB_DISPERSION
        ),

        "macro_poisson_deviance": float(
            BEST_NB_RESULT[
                "macro_metrics"
            ]["poisson_deviance"]
        ),

        "pooled_poisson_deviance": float(
            BEST_NB_RESULT[
                "pooled_metrics"
            ]["poisson_deviance"]
        ),
    },

    "compact_statistical_features": {
        "numeric":
            COMPACT_NUMERIC_FEATURES,

        "categorical":
            COMPACT_CATEGORICAL_FEATURES,

        "all":
            COMPACT_FEATURES,
    },

    "diagnostic_notes": [
        (
            "Selected Poisson model is strongly "
            "regularized and well calibrated overall."
        ),
        (
            "OOF predictions occupy a narrow range, "
            "indicating limited match-level discrimination."
        ),
        (
            "Fold 2 remains the hardest temporal regime."
        ),
        (
            "Negative Binomial dispersion did not improve "
            "expected-goal prediction."
        ),
        (
            "Final 2025-2026 test was not used for "
            "model fitting or hyperparameter selection."
        ),
    ],
}


baseline_spec_path = (
    ARTIFACT_DIR
    / "baseline_model_spec.json"
)


with open(
    baseline_spec_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        baseline_model_spec,
        f,
        indent=2,
    )


# Save genuine out-of-fold predictions separately.
baseline_oof_path = (
    ARTIFACT_DIR
    / "poisson_baseline_oof.csv"
)


BEST_POISSON_RESULT[
    "oof"
].to_csv(
    baseline_oof_path,
    index=False,
)


# Verify both artifacts immediately.
with open(
    baseline_spec_path,
    "r",
    encoding="utf-8",
) as f:

    saved_spec = json.load(f)


saved_oof = pd.read_csv(
    baseline_oof_path
)


print("PHASE 6 ARTIFACTS SAVED")
print("=" * 70)

print(
    "Model spec :",
    baseline_spec_path
)

print(
    "OOF preds  :",
    baseline_oof_path
)

print(
    "\nSelected model:",
    saved_spec[
        "selected_statistical_baseline"
    ]["model"]
)

print(
    "Feature set:",
    saved_spec[
        "selected_statistical_baseline"
    ]["feature_set"]
)

print(
    "Alpha:",
    saved_spec[
        "selected_statistical_baseline"
    ]["alpha"]
)

print(
    "Macro deviance:",
    round(
        saved_spec[
            "selected_statistical_baseline"
        ]["macro_metrics"][
            "poisson_deviance"
        ],
        6,
    )
)

print(
    "Saved OOF rows:",
    len(saved_oof)
)

print(
    "Saved OOF unique matches:",
    saved_oof[
        "match_id"
    ].nunique()
)


assert (
    saved_spec[
        "selected_statistical_baseline"
    ]["alpha"]
    == 50.0
)

assert len(saved_oof) == 340

assert (
    saved_oof["match_id"].nunique()
    == 340
)

print(
    "\nPhase 6 artifact integrity: PASSED"
)

PHASE 6 ARTIFACTS SAVED
Model spec : ..\artifacts\baseline_model_spec.json
OOF preds  : ..\artifacts\poisson_baseline_oof.csv

Selected model: PoissonRegressor
Feature set: no_season
Alpha: 50.0
Macro deviance: 1.113978
Saved OOF rows: 340
Saved OOF unique matches: 340

Phase 6 artifact integrity: PASSED
